In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch.nn as nn

import os
import zipfile
import urllib.request

## downloading data

In [ ]:
folder_name = "ml-1m"
zip_path = "ml-1m.zip"
data_url = "http://files.grouplens.org/datasets/movielens/ml-1m.zip"

if not os.path.exists(folder_name):
    print("there is no data. downloading")
    
    urllib.request.urlretrieve(data_url, zip_path)
    
    with zipfile.ZipFile(zip_path, "r") as zip_data:
        zip_data.extractall(".")
        
        os.remove(zip_path)
        print('data is downloaded')
else:
    print('data already exists')

In [ ]:
file_path = 'ml-1m/ratings.dat'
col_names = ['user_id', 'item_id', 'rating', 'timestamp']

df = pd.read_csv(file_path, sep='::', engine='python', names=col_names)
df.head()

## eda

In [ ]:
print(f'interactions quantity: {df.shape}')
print(f'unique users: {len(df['user_id'].unique())}')
print(f'unique items: {len(df['item_id'].unique())}')

In [ ]:
def rating_quantity(main_data, group_by_col, func_col, func):
    group_df = main_data.groupby(group_by_col).agg(
        quantity=(func_col, func)
    ).reset_index()
    
    group_df = group_df.rename(columns={
        'quantity':f'{func_col}_quantity'
    })
    
    group_df = group_df.sort_values(by=f'{func_col}_quantity', ascending=False)
    
    return group_df

user_rating = rating_quantity(df, 'user_id', 'rating', 'size')
item_rating = rating_quantity(df, 'item_id', 'rating', 'size')

In [ ]:
item_rating.head(10)

In [ ]:
user_rating.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=user_rating,
    x='rating_quantity'
)

plt.title('Распределение количества оценок у пользователей')
plt.xlabel('Количество поставленных оценок')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=item_rating,
    x='rating_quantity'
)

plt.title('Распределение количества оценок у фильмов')
plt.xlabel('Количество поставленных оценок')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
min_date = df['timestamp'].min()

df['last_watch_dt'] = df['timestamp'] - min_date
df['last_watch_dt'] = df['last_watch_dt'].apply(lambda x: int(str(x).split()[0]))
df = df.drop(columns='timestamp')

In [ ]:
df['last_watch_dt'].hist(bins=50, grid=False)
plt.show()

## filtering valid data

In [ ]:
active_users = user_rating.loc[user_rating['rating_quantity'] >= 10, 'user_id'].unique()
active_films = item_rating.loc[item_rating['rating_quantity'] >= 10, 'item_id'].unique()

mask = (df['user_id'].isin(active_users)) & (df['item_id'].isin(active_films))
filtered_df = df[mask]

In [ ]:
print(f'old shape: {df.shape}')
print(f'filtered df shape: {filtered_df.shape}')

In [ ]:
max_date = filtered_df['last_watch_dt'].max()
log_size = 30

cut_off_date = max_date - log_size 

## train test split

In [ ]:
train_df = filtered_df[filtered_df['last_watch_dt'] < cut_off_date]
test_df = filtered_df[filtered_df['last_watch_dt'] >= cut_off_date]

In [ ]:
assert train_df['last_watch_dt'].max() < test_df['last_watch_dt'].min(), 'wrong cut off date'

In [ ]:
train_users_ids = train_df['user_id'].unique()
train_item_ids = train_df['item_id'].unique()

test_df = test_df[test_df['user_id'].isin(train_users_ids)]
test_df = test_df[test_df['item_id'].isin(train_item_ids)]

In [ ]:
train_df.shape, test_df.shape